In [ ]:
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import ICRS    
from astropy.coordinates import Galactic 
from astropy.coordinates import SkyCoord 
import RMtools_3D.RMpeakfit_3D as rmpeakfit
import matplotlib.pyplot as plt
import astropy.units as u
import gc

In [ ]:
def gaussian(x, mu, sig):
    return np.exp(-np.power(x - mu, 2.) / (2 * np.power(sig, 2.)))

In [ ]:
def subtract_peaks(cube, fd, fwhm):

    cube_prev = cube.copy()
    cube_this = np.empty_like(cube_prev)

    pi_max = np.nanmax(cube, axis=0)

    peak_fd = np.empty_like(pi_max)
    peak_pi = np.empty_like(pi_max)
    
    pi_max = np.expand_dims(pi_max, axis=0)
    print(pi_max.shape)
    
    idx = np.abs(cube - pi_max).argmin(axis=0)
    print(idx.shape)

    for i in range(0,cube_prev.shape[1]):
        #print(i)
        for j in range(0,cube_prev.shape[2]):

            peak_fd[i,j] = fd[idx[i,j]]
            rmsf_sub = gaussian(fd, peak_fd[i,j], fwhm[i,j]/2.355)*pi_max[0,i,j]
            cube_this[:,i,j] = cube_prev[:,i,j] - rmsf_sub
            peak_pi[i,j] = pi_max[0,i,j]
    
    return peak_fd, peak_pi, cube_this

### Read in original clean cube

In [ ]:
hdu_gmims  = fits.open('/srv/data/gmims/gmims-hbn/GMIMS-HBN_v1_gal_car_FD_PI.fits')
#hdu_gmims  = fits.open('/srv/data/cgps-gmims/gmims_FD/FDF_tot_dirty.fits')
cube_gmims = hdu_gmims[0].data
hdr_gmims  = hdu_gmims[0].header

wcs_gmims = WCS(hdr_gmims)

l  = wcs_gmims.all_pix2world(range(wcs_gmims.array_shape[2]),0,0,0)[0]
b  = wcs_gmims.all_pix2world(0,range(wcs_gmims.array_shape[1]),0,0)[1]
fd = wcs_gmims.all_pix2world(0,0,range(wcs_gmims.array_shape[0]),0)[2]

hdu_rmsf = fits.open('/srv/data/cgps-gmims/gmims_FD/ordered_peaks/fwhm_rmsf.fits')
rmsf = hdu_rmsf[0].data

print(wcs_gmims)
wcs2D = wcs_gmims.dropaxis(2)
print('')
print(wcs2D)

### Determine first and second peaks

In [ ]:
fd_peak1, pi_peak1, cube_sub1 = subtract_peaks(cube_gmims, fd, rmsf)
fd_peak2, pi_peak2, cube_sub2 = subtract_peaks(cube_sub1, fd, rmsf)


### Filter first peak and check map

In [ ]:
fd_peak1_filt = fd_peak1.copy()
fd_peak1_filt[pi_peak1 < 0.03] = np.nan

fig,ax = plt.subplots(1,1,figsize=(8,6))
ax.imshow(fd_peak1_filt, cmap='RdBu_r', vmin=-80, vmax=80, origin='lower')

### Filter second peak and check map

In [ ]:
fd_peak2_filt = fd_peak2.copy()
fd_peak2_filt[pi_peak2 < 0.03] = np.nan

fig,ax = plt.subplots(1,1,figsize=(8,6))
ax.imshow(fd_peak2_filt, cmap='RdBu_r', vmin=-80, vmax=80, origin='lower')

### Make plots to show potential leakage in FD peaks and their PI values

In [ ]:
lmax = 240
lmin = 180.0001

crange = SkyCoord([lmax, lmin], [-10,10], frame=Galactic, unit=(u.deg, u.deg))

cmap=plt.cm.get_cmap('RdBu_r')
cmap.set_bad(color="grey")

fig = plt.figure(figsize=(10,12))
plt.subplots_adjust(left=0.08,    # Left margin
                   right=0.95,   # Right margin
                   bottom=0.05,  # Bottom margin
                   top=0.95,     # Top margin
                   hspace=0.1)  # Height spacing between subplots

ax1 = fig.add_subplot(411, projection=wcs2D.celestial)
ax2 = fig.add_subplot(412, projection=wcs2D.celestial)
ax3 = fig.add_subplot(413, projection=wcs2D.celestial)
ax4 = fig.add_subplot(414, projection=wcs2D.celestial)

im1 = ax1.imshow(pi_peak1, origin='lower',cmap='viridis',vmin=0,vmax=0.4)
im2 = ax2.imshow(cube_gmims[100], origin='lower',cmap='viridis',vmin=0,vmax=0.4)
im3 = ax3.imshow(fd_peak1_filt,origin='lower',cmap=cmap,vmin=-50,vmax=50)
im4 = ax4.imshow(fd_peak2_filt,origin='lower',cmap=cmap,vmin=-50,vmax=50)

axs = [ax1, ax2, ax3, ax4]

plt.colorbar(im1, ax=ax1, label=r'PI of Peak 1', fraction=0.05, pad=0.01)
plt.colorbar(im2, ax=ax2, label=r'PI of FD = 0 rad m$^{-2}$', fraction=0.05, pad=0.01)
plt.colorbar(im3, ax=ax3, label=r'FD Peak 1 (rad m$^{-2}$)', fraction=0.05, pad=0.01)
plt.colorbar(im4, ax=ax4, label=r'FD Peak 2 (rad m$^{-2}$)', fraction=0.05, pad=0.01)

for i in range(0,4):
    axs[i].set_ylim(wcs2D.world_to_pixel(crange)[1])
    axs[i].set_xlim(wcs2D.world_to_pixel(crange)[0])
    axs[i].set_ylabel('latitude')
    axs[i].set_xlabel(' ')
    axs[i].axhline(y=wcs2D.world_to_pixel(SkyCoord(0,-3,frame=Galactic,unit=(u.deg,u.deg)))[1],
                   linestyle='dashed',color='k')
    axs[i].axhline(y=wcs2D.world_to_pixel(SkyCoord(0, 5,frame=Galactic,unit=(u.deg,u.deg)))[1],
                   linestyle='dashed',color='k')
axs[3].set_xlabel('longitude')
axs[0].set_title('GMIMS-HBN, PI threshold: 0.03 K')

plt.savefig('../plots/review_tests/hbn_pi_peaks_'+str(int(lmin))+'_'+str(int(lmax))+'.png')

### Check of peak subtraction

In [ ]:
lpick = 90
bpick = 25

idxl = abs(l - lpick).argmin()
idxb = abs(b - bpick).argmin()

print(l[idxl],b[idxb])

fig,ax = plt.subplots(1,1,figsize=(12,4))

ax.plot(fd, cube_gmims[:,idxb,idxl])
ax.axvline(x=fd_peak1[idxb,idxl],color='C0',linestyle='dashed')

ax.plot(fd, cube_sub1[:,idxb,idxl])
ax.axvline(x=fd_peak2[idxb,idxl],color='C1',linestyle='dashed')

ax.set_xlim(-400,400)
#ax.set_ylim(0,)
ax.grid()

## Check things in frequency space, comparing to Stokes I

### Read in data

In [ ]:
hdu_iqu = fits.open('/srv/data/gmims/gmims-hbn/GMIMS-HBN_v1_gal_car_freq_IQU.fits')
hdr_iqu = hdu_iqu[0].header

iqu = hdu_iqu[0].data
print(iqu.shape)

wcs_iqu = WCS(hdr_iqu).dropaxis(3)

l    = wcs_iqu.all_pix2world(range(wcs_iqu.array_shape[2]),0,0,0)[0]
b    = wcs_iqu.all_pix2world(0,range(wcs_iqu.array_shape[1]),0,0)[1]
freq = wcs_iqu.all_pix2world(0,0,range(wcs_iqu.array_shape[0]),0)[2]/1e6

pi = np.sqrt(iqu[1]**2 + iqu[2]**2)
pa = 0.5*np.arctan2(iqu[2], iqu[1])

wcs2D = wcs_iqu.dropaxis(2)

In [ ]:
fpick = 1420
fidx = abs(freq - fpick).argmin()
print(freq[fidx])

plt.imshow(iqu[0][fidx],origin='lower',vmin=0,vmax=2)

In [ ]:
plt.imshow(pi[fidx],origin='lower',vmin=0,vmax=1)

In [ ]:
lmax = 240
lmin = 180.0001

crange = SkyCoord([lmax, lmin], [-10,10], frame=Galactic, unit=(u.deg, u.deg))

fig = plt.figure(figsize=(10,12))
plt.subplots_adjust(left=0.08,    # Left margin
                   right=0.95,   # Right margin
                   bottom=0.05,  # Bottom margin
                   top=0.95,     # Top margin
                   hspace=0.1)  # Height spacing between subplots

ax1 = fig.add_subplot(411, projection=wcs2D.celestial)
ax2 = fig.add_subplot(412, projection=wcs2D.celestial)
ax3 = fig.add_subplot(413, projection=wcs2D.celestial)
ax4 = fig.add_subplot(414, projection=wcs2D.celestial)

im1 = ax1.imshow(iqu[0][fidx], origin='lower',cmap='viridis',vmin=0,vmax=5)
#im2 = ax2.imshow(pi[fidx], origin='lower',cmap='viridis',vmin=0,vmax=1)
#im3 = ax3.imshow(pi[fidx]/iqu[0][fidx],origin='lower',cmap='viridis',vmin=0,vmax=1)
im2 = ax2.imshow(iqu[1][fidx], origin='lower',cmap='RdBu_r',vmin=-0.5,vmax=0.5)
im3 = ax3.imshow(iqu[2][fidx], origin='lower',cmap='RdBu_r',vmin=-0.5,vmax=0.5)
im4 = ax4.imshow(pi[fidx]/iqu[0][fidx],origin='lower',cmap='viridis',vmin=0,vmax=0.5)

axs = [ax1, ax2, ax3, ax4]

plt.colorbar(im1, ax=ax1, label=r'Stokes I', fraction=0.05, pad=0.01)
plt.colorbar(im2, ax=ax2, label=r'Stokes Q', fraction=0.05, pad=0.01)
plt.colorbar(im3, ax=ax3, label=r'Stokes U', fraction=0.05, pad=0.01)
plt.colorbar(im4, ax=ax4, label=r'PI/I', fraction=0.05, pad=0.01)

for i in range(0,4):
    axs[i].set_ylim(wcs2D.world_to_pixel(crange)[1])
    axs[i].set_xlim(wcs2D.world_to_pixel(crange)[0])
    axs[i].set_ylabel('latitude')
    axs[i].set_xlabel(' ')
    axs[i].axhline(y=wcs2D.world_to_pixel(SkyCoord(0,-3,frame=Galactic,unit=(u.deg,u.deg)))[1],
                   linestyle='dashed',color='k')
    axs[i].axhline(y=wcs2D.world_to_pixel(SkyCoord(0, 5,frame=Galactic,unit=(u.deg,u.deg)))[1],
                   linestyle='dashed',color='k')
axs[3].set_xlabel('longitude')
axs[0].set_title('GMIMS-HBN, '+str(freq[fidx])+' MHz')

plt.savefig('../plots/review_tests/hbn_i_and_pi_'+str(int(lmin))+'_'+str(int(lmax))+'.png')

## PI across the 35-MHz band

In [ ]:
def read_files_4channels(directory,stokes,filetype):

    band = ['A','B','C','D']
    data_list = []
    hdr_list = []
    print('Reading in '+stokes+' for filetype: '+filetype)

    for i in range(0,4):
        print('band '+band[i])
        hdu = fits.open(directory+stokes+band[i]+'_'+filetype+'.fits')
        data_list.append(hdu[0].data)

        hdr = fits.Header()
        for card in hdu[0].header.cards:
            if card.keyword.strip() != "":
                hdr.append(card)
        hdr['OBJECT'] = stokes+band[i]+'_'+filetype
        hdr_list.append(hdr)
        #print(repr(hdr))
        #print('-------------------')

    gc.collect()

    return data_list,hdr_list

In [ ]:
P_thr   = 0.1 # K
dRM_thr = 150 # rad/m^2

In [ ]:
hdu_PI_CG = fits.open('/srv/data/cgps-gmims/conv_regrid/PI_CG_conv4_regrd_PI_of_mean.fits')
PI_CG     = hdu_PI_CG[0].data

In [ ]:
q_data_list, q_hdr_list = read_files_4channels('/srv/data/cgps-gmims/conv_regrid/','Q','CG_conv4_regrd')
u_data_list, u_hdr_list = read_files_4channels('/srv/data/cgps-gmims/conv_regrid/','U','CG_conv4_regrd')


In [ ]:
pi_list = []

for i in range(0,4):
    pi_list.append(np.sqrt(q_data_list[i]**2 + u_data_list[i]**2))
    

In [ ]:
ratio = pi_list[0].flatten()/pi_list[3].flatten()

ratio_masked = ratio.copy()
ratio_masked[PI_CG.flatten() < P_thr] = np.nan

num1 = len(np.where( ratio > 1.05 )[0]) # above 5%
num2 = len(np.where( ratio < 1.0 )[0])  # negative diff (low freq has lower PI)
num3 = len(np.where( (ratio <= 1.05) & (ratio >= 1.0) )[0])
numtot = num1+num2+num3
print(numtot)

num1_masked = len(np.where( ratio_masked > 1.05 )[0]) # above 5%
num2_masked = len(np.where( ratio_masked < 1.0 )[0])  # negative diff (low freq has lower PI)
num3_masked = len(np.where( (ratio_masked <= 1.05) & (ratio_masked >= 1.0) )[0])
numtot_masked = num1_masked+num2_masked+num3_masked
print(numtot_masked)

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(12,4))
plt.subplots_adjust(left=0.08,    # Left margin
                   right=0.98,   # Right margin
                   bottom=0.15,  # Bottom margin
                   top=0.98,     # Top margin
                   )  

ax.hist(ratio, bins=200, range=(0,3),alpha=0.5, label='All data, no thresholding');
ax.hist(ratio_masked, bins=200, range=(0,3),alpha=0.5, label='With PI threshold 0.1 K');
ax.set_xlim(0,3)
ax.axvline(x=1.05, color='k', linestyle='dashed')
ax.axvline(x=1, color='k', linestyle='dashed')

ax.text(2,2.5e5,r'P$_{1403}$/P$_{1438}$ < 1.0 : '+str(np.round(100*num2/numtot,1))+'% ('+str(np.round(100*num2_masked/numtot_masked,1))+'%)')
ax.text(1.83,2e5,r'1.0 < P$_{1403}$/P$_{1438}$ < 1.05 : '+str(np.round(100*num3/numtot,1))+'% ('+str(np.round(100*num3_masked/numtot_masked,1))+'%)')
ax.text(2,1.5e5,r'P$_{1403}$/P$_{1438}$ > 1.05 : '+str(np.round(100*num1/numtot,1))+'% ('+str(np.round(100*num1_masked/numtot_masked,1))+'%)')

ax.set_xlabel(r'P$_{1403}$/P$_{1438}$ ratio')
ax.legend()

plt.savefig('../plots/review_tests/PI_ratios_histogram.png')

In [ ]:
wcs2D = WCS(q_hdr_list[0])
print(wcs2D)

In [ ]:
lmax = 120
lmin = 90

crange = SkyCoord([lmax, lmin], [-6,6], frame=Galactic, unit=(u.deg, u.deg))

fig = plt.figure(figsize=(10,12))
plt.subplots_adjust(left=0.08,    # Left margin
                   right=0.95,   # Right margin
                   bottom=0.05,  # Bottom margin
                   top=0.95,     # Top margin
                   hspace=0.1)  # Height spacing between subplots

ax1 = fig.add_subplot(311, projection=wcs2D.celestial)
ax2 = fig.add_subplot(312, projection=wcs2D.celestial)
ax3 = fig.add_subplot(313, projection=wcs2D.celestial)

#im1 = ax1.imshow(pi_mean, origin='lower',cmap='viridis',vmin=0,vmax=0.5)
im1 = ax1.imshow(pi_list[0]/pi_list[3], origin='lower',cmap='RdBu_r',vmin=0.5,vmax=1.5)
im2 = ax2.imshow(pi_list[0] - pi_list[3], origin='lower',cmap='RdBu_r',vmin=-0.5,vmax=0.5)
im3 = ax3.imshow((pi_list[0] - pi_list[3])/pi_mean, origin='lower',cmap='RdBu_r',vmin=-0.5,vmax=0.5)

axs = [ax1, ax2, ax3]

plt.colorbar(im1, ax=ax1, label=r' ', fraction=0.05, pad=0.01)
plt.colorbar(im2, ax=ax2, label=r' ', fraction=0.05, pad=0.01)
plt.colorbar(im3, ax=ax3, label=r' ', fraction=0.05, pad=0.01)

for i in range(0,3):
    axs[i].set_ylim(wcs2D.world_to_pixel(crange)[1])
    axs[i].set_xlim(wcs2D.world_to_pixel(crange)[0])
    axs[i].set_ylabel('latitude')
    axs[i].set_xlabel(' ')
    axs[i].axhline(y=wcs2D.world_to_pixel(SkyCoord(0,-3,frame=Galactic,unit=(u.deg,u.deg)))[1],
                   linestyle='dashed',color='k')
    axs[i].axhline(y=wcs2D.world_to_pixel(SkyCoord(0, 5,frame=Galactic,unit=(u.deg,u.deg)))[1],
                   linestyle='dashed',color='k')
axs[2].set_xlabel('longitude')
#axs[0].set_title('GMIMS-HBN, '+str(freq[fidx])+' MHz')

In [ ]:
np.exp(-2.8)

In [ ]:
plt.scatter(pi_list[0][pi_mean>0.2].flatten(), pi_list[3][pi_mean>0.2].flatten(), s=5)
plt.plot([0,8],[0,8],linestyle='dashed',color='k')

### Simulated LOS to check for possible scenarios of PI changing across the band

In [ ]:
from faraday_astro_sim import make_los

Example scenarios of changes across 35 MHz band:
 1. No change in PI across the band (or any band): single FD screen
 2. Lower PI for lower frequency (opposite to spectral index): Burn slab, width 50 rad/m^2
 3. Lower PI for lower frequency (same way as spectral index): Burn slab, width 80 rad/m^2
 4. RM over 35 MHz band loses meaning: two screens, unequal PI

In [ ]:
parameters = {"pi"  : [0.5, 0.5],
              "phi" : [20, 60],
              "psi" : [10.0, 10.0],
              "sig" : [0.0, 0.0],
              "dphi": [0.0, 0.0]}

freqs = np.arange(500e6,5000e6,1e6)
results = make_los(parameters, freqs=freqs, noise=0.000001)


In [ ]:
fig, ax = plt.subplots(2,1,figsize=(12,6))
plt.subplots_adjust(left=0.08,    # Left margin
                   right=0.93,   # Right margin
                   bottom=0.13,  # Bottom margin
                   top=0.98,     # Top margin
                   hspace=0.3)  # Height spacing between subplots

p1403 = abs(results['pol'][abs(results['freq']-1403e6).argmin()])
p1438 = abs(results['pol'][abs(results['freq']-1438e6).argmin()])

polangle = 0.5*np.arctan2(results['pol'].imag, results['pol'].real)

ax[0].scatter(results['lsq'],abs(results['pol']), s=0.5)
#ax.scatter(results['lsq'],results['pol'].real, s=0.5)
#ax.scatter(results['lsq'],results['pol'].imag, s=0.5)
ax[0].set_xlim(0,0.15)
ax[0].set_ylim(0,1.1)
ax[0].axvline(x=(3e8/1403e6)**2,color='k',linewidth=1,linestyle='dashed')
ax[0].axvline(x=(3e8/1438e6)**2,color='k',linewidth=1,linestyle='dashed')

ax2 = ax[0].twinx()
ax2.scatter(results['lsq'], polangle, color='red', s=0.5, alpha=0.5)
ax2.set_ylim(-np.pi/2,np.pi/2)

ax[0].text(0.11,0.8,r'P$_{1403}$ = '+str(np.round(p1403,3)),fontsize=16)
ax[0].text(0.11,0.65,r'P$_{1438}$ = '+str(np.round(p1438,3)),fontsize=16)
ax[0].text(0.11,0.5,r'P$_{1403}$/P$_{1438}$ = '+str(np.round(p1403/p1438,3)),fontsize=16)
ax[0].text(0.11,0.35,r'change = '+str(np.round((p1403/p1438-1)*100,1))+'%',fontsize=16)

ax[0].set_xlabel(r'$\lambda^2$ (m$^2$)')
ax[0].set_ylabel(r'Polarised intensity')
ax2.set_ylabel(r'Polarisation angle (rad)')

ax[1].scatter(results['fdf_dirty'][1]['phiArr_radm2'], abs(results['fdf_dirty'][1]['dirtyFDF']),s=0.5)
ax[1].set_ylim(0,)
ax[1].set_xlim(-200,200)
ax[1].grid()

ax[1].set_xlabel(r'Faraday depth (rad m$^{-2}$)')
ax[1].set_ylabel(r'Polarised intensity')

#plt.savefig('../plots/review_tests/sim_spectra_lowfreq_highPI.png')